# Task 2 — SID data contract

Run `00_colab_setup.ipynb` first. Set its `DATA_SUBDIRECTORY` to `raw/sid_set`, set `COPY_DATA = True`, and complete the strict smoke check before running this notebook.

In [4]:
from pathlib import Path
import json
import shutil
import subprocess

PROJECT_ROOT = Path("/content/cya-techjam26")
DATA_ROOT = Path("/content/hackathon_data")
RAW_SID_ROOT = DATA_ROOT / "raw/sid_set"
TASK2_ROOT = PROJECT_ROOT / "artifacts/task2"
DRIVE_TASK2_ROOT = Path("/content/drive/MyDrive/cya-techjam26/artifacts/task2")

assert (PROJECT_ROOT / "configs/colab.json").is_file(), "Remote repository is not ready"
assert (RAW_SID_ROOT / "labels.csv").is_file(), f"Missing {RAW_SID_ROOT / 'labels.csv'}"
assert (RAW_SID_ROOT / "images").is_dir(), f"Missing {RAW_SID_ROOT / 'images'}"
TASK2_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_TASK2_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Raw SID data: {RAW_SID_ROOT}")

Raw SID data: /content/hackathon_data/raw/sid_set


## 1. Audit immutable sources

This verifies 20,000 CSV rows, excludes tampered rows, checks corruption, records C2PA status, hashes files, and groups exact/near duplicates. Do not use `--skip-c2pa` for this canonical run.

In [5]:
subprocess.run(
    ["make", "task2-source-audit", f"DATA_ROOT={DATA_ROOT}", f"ARTIFACT_ROOT={PROJECT_ROOT / 'artifacts'}"],
    cwd=PROJECT_ROOT,
    check=True,
)
source_audit = json.loads((TASK2_ROOT / "source_audit.json").read_text())
source_audit

{'c2pa_status_counts': {'no_manifest': 20000},
 'corrupt_count': 0,
 'cross_label_duplicate_groups': 0,
 'csv_rows': 20000,
 'dataset_name': 'sid_set',
 'dataset_root': '/content/hackathon_data/raw/sid_set',
 'duplicate_groups': 106,
 'eligible_primary_count': 19882,
 'excluded_label_counts': {},
 'included_binary_rows': 20000,
 'label_counts': {'ai_generated': 10000, 'authentic': 10000},
 'perceptual_hamming_distance': 4,
 'unlisted_image_count': 0,
 'unlisted_images_preview': []}

In [6]:
assert source_audit["csv_rows"] == 20000
assert source_audit["corrupt_count"] == 0
assert source_audit["cross_label_duplicate_groups"] == 0
assert source_audit["c2pa_status_counts"].get("dependency_missing", 0) == 0
assert source_audit["c2pa_status_counts"].get("scan_error", 0) == 0
print("Source audit gates passed.")

Source audit gates passed.


## 2. Freeze grouped splits and audit native nuisance signals

In [7]:
subprocess.run(
    ["make", "task2-split", f"ARTIFACT_ROOT={PROJECT_ROOT / 'artifacts'}"],
    cwd=PROJECT_ROOT,
    check=True,
)
subprocess.run(
    ["make", "task2-nuisance-source", f"ARTIFACT_ROOT={PROJECT_ROOT / 'artifacts'}"],
    cwd=PROJECT_ROOT,
    check=True,
)
split_report = json.loads((TASK2_ROOT / "split_report.json").read_text())
native_nuisance = json.loads((TASK2_ROOT / "source_nuisance.json").read_text())
print(json.dumps(split_report["counts"], indent=2, sort_keys=True))
print(json.dumps({k: native_nuisance[k] for k in ("test_accuracy", "test_balanced_accuracy", "test_roc_auc")}, indent=2))

{
  "final_test:ai_generated": 741,
  "final_test:authentic": 750,
  "seed_train:ai_generated": 5929,
  "seed_train:authentic": 6000,
  "selection_val:ai_generated": 741,
  "selection_val:authentic": 750,
  "self_train_pool:ai_generated": 2471,
  "self_train_pool:authentic": 2500
}
{
  "test_accuracy": 0.6767812238055323,
  "test_balanced_accuracy": 0.6770140528386734,
  "test_roc_auc": 0.7369459246767847
}


## 3. Build equal-label matched-clean pilots

Each policy pilot uses at most 1,000 primary sources per label, applies the same encoding policy to both labels, preserves dimensions, strips metadata, and uses JPEG 4:4:4.

In [8]:
subprocess.run(
    ["make", "task2-pilots", f"ARTIFACT_ROOT={PROJECT_ROOT / 'artifacts'}"],
    cwd=PROJECT_ROOT,
    check=True,
)

comparison = {}
for name in ("fixed_q96", "uniform_q95_q100"):
    report = json.loads((TASK2_ROOT / f"{name}_nuisance.json").read_text())
    comparison[name] = {
        key: report[key]
        for key in ("sample_count", "test_accuracy", "test_balanced_accuracy", "test_roc_auc")
    }
comparison

{'fixed_q96': {'sample_count': 2000,
  'test_accuracy': 0.6483333333333333,
  'test_balanced_accuracy': 0.6483333333333333,
  'test_roc_auc': 0.7098333333333333},
 'uniform_q95_q100': {'sample_count': 2000,
  'test_accuracy': 0.6366666666666667,
  'test_balanced_accuracy': 0.6366666666666667,
  'test_roc_auc': 0.6921611111111111}}

Do not choose a policy from nuisance accuracy alone. Preserve both pilot reports for the Stage A matched-policy comparison; the selected policy must also retain clean and robustness performance.

In [9]:
for path in TASK2_ROOT.iterdir():
    if path.is_file() and path.suffix in {".json", ".csv"}:
        shutil.copy2(path, DRIVE_TASK2_ROOT / path.name)
print(f"Copied Task 2 manifests/reports to {DRIVE_TASK2_ROOT}")

Copied Task 2 manifests/reports to /content/drive/MyDrive/cya-techjam26/artifacts/task2
